# True/False Generator

Schema-driven · Batch-based · Judge-validated

Each True/False question has:
- `context` — a short reading passage (1-3 sentences)
- `statement` — a statement about the context for the student to judge
- `correct_answer` — `"True"` or `"False"`
- `justification` — one sentence explaining why the answer is True or False
- `explanation` — teaching note on the language point tested

**CEFR sub-types:**
| Level | Task type |
|---|---|
| A1/A2 | Explicit fact check — answer is directly stated in the context |
| B1/B2 | Implicit inference — answer requires reading between the lines |
| C1/C2 | Critical evaluation — answer depends on nuance, register, or implication |

**Output:** `data/true_false/true_false_generator_output.json`

In [ ]:
from __future__ import annotations
import json, sys
from pathlib import Path
from typing import Literal

import dspy
from dotenv import load_dotenv
from pydantic import BaseModel, Field

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / '.env')

from utils import configure_lm
print('Project root:', PROJECT_ROOT)

In [ ]:
lm = configure_lm()
dspy.configure(lm=lm)
print('LM:', lm.model)

In [ ]:
ARTIFACTS           = PROJECT_ROOT / 'artifacts'
DIFFICULTY_ARTIFACT = ARTIFACTS / 'simple_difficulty_optimized.json'
RUBRIC_ARTIFACT     = ARTIFACTS / 'rubric_agent_optimized.json'
OUTPUT_PATH         = PROJECT_ROOT / 'data' / 'true_false' / 'true_false_generator_output.json'
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
class SubtopicRequirement(BaseModel):
    subtopic:  str
    a1_count:  int = 0
    a2_count:  int = 0
    b1_count:  int = 0
    b2_count:  int = 0
    c1_count:  int = 0
    c2_count:  int = 0

    @property
    def easy_count(self) -> int:   return self.a1_count + self.a2_count
    @property
    def medium_count(self) -> int: return self.b1_count + self.b2_count
    @property
    def hard_count(self) -> int:   return self.c1_count + self.c2_count
    @property
    def total(self) -> int:        return self.easy_count + self.medium_count + self.hard_count

class GenerationConstraints(BaseModel):
    questions_per_iteration: int   = 3
    max_iterations:          int   = 10
    difficulty_match:        bool  = True
    rubric_min_score:        float = 0.7
    true_false_balance:      bool  = True   # try to keep ~50% True and ~50% False

class InputSchema(BaseModel):
    topic:       str
    subtopics:   list[SubtopicRequirement]
    constraints: GenerationConstraints = Field(default_factory=GenerationConstraints)

In [ ]:
class TrueFalseItem(BaseModel):
    question_number:   int
    topic:             str
    subtopic:          str | None
    target_cefr:       str
    target_difficulty: str
    task_type:         str   # 'explicit', 'inference', 'critical'
    instruction:       str
    context:           str   # short reading passage
    statement:         str   # T/F statement about the context
    correct_answer:    Literal['True', 'False']
    justification:     str   # one sentence explaining why
    explanation:       str   # teaching note


class TrueFalseQuestionStore(BaseModel):
    easy:   list[TrueFalseItem] = Field(default_factory=list)
    medium: list[TrueFalseItem] = Field(default_factory=list)
    hard:   list[TrueFalseItem] = Field(default_factory=list)

    def add(self, item: TrueFalseItem) -> None:
        d = item.target_difficulty.strip().lower()
        if d == 'easy':   self.easy.append(item)
        elif d == 'medium': self.medium.append(item)
        elif d == 'hard':   self.hard.append(item)

    def count(self, difficulty: str) -> int:
        return len(getattr(self, difficulty.lower(), []))

    @property
    def total(self) -> int:
        return len(self.easy) + len(self.medium) + len(self.hard)

In [ ]:
# Task type per CEFR
_CEFR_TASK_TYPE = {
    'A1': 'explicit',   'A2': 'explicit',
    'B1': 'inference',  'B2': 'inference',
    'C1': 'critical',   'C2': 'critical',
}


class ExampleTrueFalseQuestion(BaseModel):
    instruction:    str
    context:        str
    task_type:      str
    statement:      str
    correct_answer: str
    justification:  str
    explanation:    str
    difficulty:     str
    cefr:           str
    subtopic:       str


class TrueFalseGenerationRequest(BaseModel):
    topic:                 str
    subtopic:              str
    target_cefr:           str
    target_difficulty:     str
    task_type:             str
    example_questions:     list[ExampleTrueFalseQuestion]
    already_used_contexts: list[str]
    batch_size:            int
    aim_for_true:          bool   # hint to balance True/False ratio


class TrueFalseGeneratedQuestion(BaseModel):
    instruction:    str
    context:        str
    statement:      str
    correct_answer: Literal['True', 'False']
    justification:  str
    explanation:    str


class TrueFalseGenerationBatch(BaseModel):
    questions: list[TrueFalseGeneratedQuestion]


class TrueFalseGeneratorSignature(dspy.Signature):
    '''Generate a batch of True/False questions for English language learners.

    Task types by CEFR:
    - explicit (A1/A2): The statement is directly confirmed or contradicted by the context text.
      Use simple vocabulary. The answer is unambiguous from a single reading.
    - inference (B1/B2): The statement requires the student to draw a conclusion not explicitly stated.
      The context hints at the answer through implication.
    - critical (C1/C2): The statement requires evaluating nuance, register shift, or implicit meaning.
      The context may use hedging language, irony, or academic qualification.

    Rules:
    - context: 1-3 natural sentences forming a coherent mini-passage
    - statement: a single declarative sentence that is clearly True or False given the context
    - correct_answer: exactly "True" or "False" (no other values)
    - justification: one sentence quoting or referencing the relevant part of the context
    - explanation: one teaching note on the language or reasoning skill tested
    - aim_for_true hint: if True, try to generate mostly True answers in this batch
    - Each context must be unique and not appear in already_used_contexts
    '''
    request: TrueFalseGenerationRequest = dspy.InputField(desc='Generation parameters and examples')
    batch:   TrueFalseGenerationBatch   = dspy.OutputField(desc='Generated True/False questions')

In [ ]:
class DifficultyResult(BaseModel):
    question_id:          str
    predicted_cefr:       Literal['A1', 'A2', 'B1', 'B2', 'C1', 'C2']
    predicted_difficulty: Literal['Easy', 'Medium', 'Hard']

class DifficultyOutput(BaseModel):
    results: list[DifficultyResult]

class TrueFalseDifficultySignature(dspy.Signature):
    '''Classify True/False questions by CEFR level.
    Consider: context vocabulary complexity, statement directness, reasoning required.
    Explicit fact-check = Easy, inference = Medium, critical evaluation = Hard.
    '''
    questions: str             = dspy.InputField(desc='JSON list of T/F questions')
    output:    DifficultyOutput = dspy.OutputField(desc='CEFR and difficulty predictions')

class DifficultyJudgeWrapper:
    def __init__(self, artifact_path: Path):
        self.judge = dspy.Predict(TrueFalseDifficultySignature)
        if artifact_path.exists(): self.judge.load(str(artifact_path))
        print(f'  DifficultyJudge: {"loaded" if artifact_path.exists() else "zero-shot"}')
    def judge_batch(self, questions):
        return self.judge(questions=json.dumps(questions)).output.results


class RubricScore(BaseModel):
    question_id:           str
    context_clarity:       float = Field(ge=0, le=1)
    statement_unambiguity: float = Field(ge=0, le=1)
    answer_defensibility:  float = Field(ge=0, le=1)
    justification_quality: float = Field(ge=0, le=1)
    cefr_appropriateness:  float = Field(ge=0, le=1)
    avg_score:             float = Field(ge=0, le=1)

class RubricOutput(BaseModel):
    scores: list[RubricScore]

class TrueFalseRubricSignature(dspy.Signature):
    '''Score True/False question quality on 5 criteria (0-1 each).
    context_clarity: passage is coherent and grammatically correct
    statement_unambiguity: only one defensible True/False answer exists
    answer_defensibility: the stated correct_answer is clearly supported by the context
    justification_quality: justification accurately references the context
    cefr_appropriateness: language complexity matches the target CEFR level
    avg_score: mean of all five scores.
    '''
    questions: str         = dspy.InputField(desc='JSON list of T/F questions')
    output:    RubricOutput = dspy.OutputField(desc='Quality scores')

class RubricJudgeWrapper:
    def __init__(self, artifact_path: Path):
        self.judge = dspy.Predict(TrueFalseRubricSignature)
        if artifact_path.exists(): self.judge.load(str(artifact_path))
        print(f'  RubricJudge: {"loaded" if artifact_path.exists() else "zero-shot"}')
    def score_batch(self, questions):
        return self.judge(questions=json.dumps(questions)).output.scores

In [ ]:
EXAMPLE_QUESTIONS: list[ExampleTrueFalseQuestion] = [
    ExampleTrueFalseQuestion(
        instruction='Read the text and decide if the statement is True or False.',
        context='The shop opens at 9 AM and closes at 5 PM.',
        task_type='explicit',
        statement='The shop closes before 6 PM.',
        correct_answer='True',
        justification='The context states the shop closes at 5 PM, which is before 6 PM.',
        explanation='A1 explicit: the answer is directly stated and requires simple comparison.',
        difficulty='Easy', cefr='A1', subtopic='daily routines'
    ),
    ExampleTrueFalseQuestion(
        instruction='Read the text and decide if the statement is True or False.',
        context='Maria went to the market this morning and bought apples, bananas, and bread. She did not buy any vegetables.',
        task_type='explicit',
        statement='Maria bought vegetables at the market.',
        correct_answer='False',
        justification='The context explicitly states she did not buy any vegetables.',
        explanation='A2 explicit: the answer is a direct negation stated in the text.',
        difficulty='Easy', cefr='A2', subtopic='shopping'
    ),
    ExampleTrueFalseQuestion(
        instruction='Read the text and decide if the statement is True or False.',
        context='Tom has been studying the same grammar topic every evening for a month, but his test scores have not improved.',
        task_type='inference',
        statement='Tom\'s study method is effective.',
        correct_answer='False',
        justification='The context implies the method is ineffective since his scores have not improved despite consistent practice.',
        explanation='B1 inference: the student must infer from the outcome that the method fails.',
        difficulty='Medium', cefr='B1', subtopic='learning strategies'
    ),
    ExampleTrueFalseQuestion(
        instruction='Read the text and decide if the statement is True or False.',
        context='Although the project deadline was extended twice, the team still submitted incomplete work. The manager praised their effort publicly but privately expressed disappointment.',
        task_type='inference',
        statement='The manager was entirely satisfied with the team\'s performance.',
        correct_answer='False',
        justification='The manager expressed disappointment privately, suggesting public praise did not reflect genuine satisfaction.',
        explanation='B2 inference: the student must distinguish between public behaviour and private feelings implied in the text.',
        difficulty='Medium', cefr='B2', subtopic='workplace communication'
    ),
    ExampleTrueFalseQuestion(
        instruction='Read the extract and decide if the statement is True or False.',
        context='The report tentatively suggests that further investment may yield modest improvements, though the authors acknowledge that the evidence base remains limited and contested.',
        task_type='critical',
        statement='The authors are confident that investment will significantly improve outcomes.',
        correct_answer='False',
        justification='Words such as "tentatively", "may", "modest", and "limited and contested" signal hedging and uncertainty, not confidence.',
        explanation='C1 critical: recognising hedging language (modal verbs, adverbs of caution) is required to evaluate the degree of certainty expressed.',
        difficulty='Hard', cefr='C1', subtopic='academic language'
    ),
    ExampleTrueFalseQuestion(
        instruction='Read the extract carefully and decide if the statement is True or False.',
        context='The legislation, ostensibly designed to protect consumers, has in practice disproportionately benefited large corporations by raising compliance costs beyond the reach of smaller competitors.',
        task_type='critical',
        statement='The legislation has achieved its stated purpose.',
        correct_answer='False',
        justification='The phrase "ostensibly designed" signals the stated purpose differs from the actual outcome described — benefiting corporations rather than consumers.',
        explanation='C2 critical: the word "ostensibly" signals ironic or critical distance between stated intention and real effect, requiring sophisticated register awareness.',
        difficulty='Hard', cefr='C2', subtopic='law and society'
    ),
]

In [ ]:
def hard_validate(q: TrueFalseGeneratedQuestion) -> tuple[bool, str]:
    if q.correct_answer not in ('True', 'False'):
        return False, f'correct_answer must be True or False, got: {q.correct_answer}'
    if not q.context.strip():
        return False, 'context is empty'
    if not q.statement.strip():
        return False, 'statement is empty'
    if not q.justification.strip():
        return False, 'justification is empty'
    return True, ''


class TrueFalseGeneratorAgent(dspy.Module):
    def __init__(self, *, store, difficulty_judge, rubric_judge):
        super().__init__()
        self.store            = store
        self.difficulty_judge = difficulty_judge
        self.rubric_judge     = rubric_judge
        self.generator        = dspy.Predict(TrueFalseGeneratorSignature)
        self._counter         = 0

    def forward(self, *, schema, example_questions, topic, subtopic,
                target_cefr, target_difficulty, target_count, rejected, warnings):
        used_contexts = [q.context for d in (self.store.easy, self.store.medium, self.store.hard)
                         for q in d]
        task_type = _CEFR_TASK_TYPE[target_cefr]
        diff_map = {'A1':'Easy','A2':'Easy','B1':'Medium','B2':'Medium','C1':'Hard','C2':'Hard'}
        iteration = 0

        # Track T/F balance
        all_answers = [q.correct_answer for d in (self.store.easy, self.store.medium, self.store.hard)
                       for q in d]

        while self.store.count(target_difficulty) < target_count and iteration < schema.constraints.max_iterations:
            iteration += 1
            true_count  = all_answers.count('True')
            false_count = all_answers.count('False')
            aim_for_true = true_count <= false_count if schema.constraints.true_false_balance else True

            req = TrueFalseGenerationRequest(
                topic=topic, subtopic=subtopic,
                target_cefr=target_cefr, target_difficulty=target_difficulty,
                task_type=task_type,
                example_questions=example_questions,
                already_used_contexts=used_contexts,
                batch_size=min(schema.constraints.questions_per_iteration,
                               target_count - self.store.count(target_difficulty) + 2),
                aim_for_true=aim_for_true,
            )
            try:
                candidates = self.generator(request=req).batch.questions
            except Exception as e:
                warnings.append(f'[{target_cefr}] iter {iteration}: {e}')
                continue

            valid = []
            for q in candidates:
                ok, reason = hard_validate(q)
                if ok: valid.append(q)
                else:  rejected.append({'cefr': target_cefr, 'reason': reason})

            if not valid: continue

            judge_input = [{'question_id': f'q{i}', 'context': q.context,
                            'statement': q.statement, 'correct_answer': q.correct_answer,
                            'target_cefr': target_cefr, 'target_difficulty': target_difficulty}
                           for i, q in enumerate(valid)]

            try:    diff_results = {r.question_id: r for r in self.difficulty_judge.judge_batch(judge_input)}
            except: diff_results = {}
            try:    rubric_results = {r.question_id: r for r in self.rubric_judge.score_batch(judge_input)}
            except: rubric_results = {}

            for i, q in enumerate(valid):
                qid = f'q{i}'
                dr, rr = diff_results.get(qid), rubric_results.get(qid)

                if dr and diff_map.get(dr.predicted_cefr) != target_difficulty:
                    rejected.append({'cefr': target_cefr, 'reason': f'Difficulty mismatch: {dr.predicted_cefr}'})
                    continue
                if rr and rr.avg_score < schema.constraints.rubric_min_score:
                    rejected.append({'cefr': target_cefr, 'reason': f'Rubric low: {rr.avg_score:.2f}'})
                    continue

                self._counter += 1
                self.store.add(TrueFalseItem(
                    question_number=self._counter, topic=topic, subtopic=subtopic,
                    target_cefr=target_cefr, target_difficulty=target_difficulty,
                    task_type=task_type, instruction=q.instruction,
                    context=q.context, statement=q.statement,
                    correct_answer=q.correct_answer, justification=q.justification,
                    explanation=q.explanation,
                ))
                used_contexts.append(q.context)
                all_answers.append(q.correct_answer)
                if self.store.count(target_difficulty) >= target_count:
                    break

In [ ]:
_CEFR_TO_DIFFICULTY = {'A1':'Easy','A2':'Easy','B1':'Medium','B2':'Medium','C1':'Hard','C2':'Hard'}
_CEFR_LEVELS = ['A1','A2','B1','B2','C1','C2']

def run_true_false_generation(schema: InputSchema) -> tuple[TrueFalseQuestionStore, list, list]:
    difficulty_judge = DifficultyJudgeWrapper(DIFFICULTY_ARTIFACT)
    rubric_judge     = RubricJudgeWrapper(RUBRIC_ARTIFACT)
    store    = TrueFalseQuestionStore()
    rejected, warnings = [], []
    agent = TrueFalseGeneratorAgent(store=store, difficulty_judge=difficulty_judge, rubric_judge=rubric_judge)

    for subtopic_req in schema.subtopics:
        for cefr in _CEFR_LEVELS:
            target_count = getattr(subtopic_req, f'{cefr.lower()}_count', 0)
            if target_count == 0: continue
            difficulty = _CEFR_TO_DIFFICULTY[cefr]
            examples   = [q for q in EXAMPLE_QUESTIONS if q.cefr == cefr]
            print(f'  {cefr} ({difficulty}) — {subtopic_req.subtopic} — task: {_CEFR_TASK_TYPE[cefr]}')
            agent.forward(
                schema=schema, example_questions=examples,
                topic=schema.topic, subtopic=subtopic_req.subtopic,
                target_cefr=cefr, target_difficulty=difficulty,
                target_count=store.count(difficulty) + target_count,
                rejected=rejected, warnings=warnings,
            )
    return store, rejected, warnings

In [ ]:
SCHEMA = InputSchema(
    topic='English Reading Comprehension',
    subtopics=[
        SubtopicRequirement(subtopic='daily life',       a1_count=1, a2_count=1),
        SubtopicRequirement(subtopic='workplace',        b1_count=1, b2_count=1),
        SubtopicRequirement(subtopic='academic writing', c1_count=1, c2_count=1),
    ],
    constraints=GenerationConstraints(questions_per_iteration=3, max_iterations=8,
                                       rubric_min_score=0.65, true_false_balance=True),
)

print('Schema:', SCHEMA.topic)
for s in SCHEMA.subtopics:
    print(f'  {s.subtopic}: total={s.total}')

store, rejected, warnings = run_true_false_generation(SCHEMA)
print(f'\nGenerated: easy={len(store.easy)} medium={len(store.medium)} hard={len(store.hard)}')
print(f'Rejected : {len(rejected)}')

# T/F balance report
all_items = store.easy + store.medium + store.hard
true_count  = sum(1 for q in all_items if q.correct_answer == 'True')
false_count = sum(1 for q in all_items if q.correct_answer == 'False')
print(f'Balance  : True={true_count}  False={false_count}')

In [ ]:
for difficulty in ('easy', 'medium', 'hard'):
    items = getattr(store, difficulty)
    if not items: continue
    print(f'\n{"="*70}  {difficulty.upper()}')
    for item in items:
        print(f'Q{item.question_number} [{item.target_cefr}] {item.task_type} — {item.subtopic}')
        print(f'  Context  : {item.context}')
        print(f'  {item.instruction}')
        print(f'  Statement: {item.statement}')
        print(f'  Answer   : {item.correct_answer}')
        print(f'  Why      : {item.justification}')
        print()

In [ ]:
output = {
    'metadata': {
        'topic': SCHEMA.topic,
        'question_type': 'true_false',
        'total': store.total,
        'rejected': len(rejected),
        'true_count': true_count,
        'false_count': false_count,
    },
    'questions': {
        'easy':   [q.model_dump() for q in store.easy],
        'medium': [q.model_dump() for q in store.medium],
        'hard':   [q.model_dump() for q in store.hard],
    },
    'rejected': rejected,
}
OUTPUT_PATH.write_text(json.dumps(output, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Saved {store.total} questions to {OUTPUT_PATH}')